<a href="https://colab.research.google.com/github/Najaf-Ali12/Complete-NLP-Projects/blob/main/Supervised_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 24.4 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

In [4]:
MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "medalpaca/medical_meadow_medical_flashcards"
NUM_SAMPLES  = 5000
OUTPUT_DIR   = "qwen-medical-full-sft"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU: Tesla T4


In [5]:
raw = load_dataset(DATASET_NAME, split="train")
print(raw)
for i in range(3):
    print("INSTRUCTION:", raw[i]["instruction"])
    print("INPUT      :", raw[i]["input"])
    print("OUTPUT     :", raw[i]["output"])
    print("-" * 60)

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…): reconstructing file:   0%|          |  0.00B / 17.7MB            

medical_meadow_wikidoc_medical_flashcard(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 33955
})
INSTRUCTION: Answer this question truthfully
OUTPUT     : Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.
------------------------------------------------------------
INSTRUCTION: Answer this question truthfully
OUTPUT     : Low estradiol production leads to genitourinary syndrome of menopause (atrophic vaginitis).
------------------------------------------------------------
INSTRUCTION: Answer this question truthfully
OUTPUT     : Low REM sleep latency and experiencing hallucinations/sleep paralysis suggests narcolepsy.
------------------------------------------------------------


## Cell 4: Convert to chat format
TRL applies the model's chat template to the `messages` column automatically.

In [6]:
# this ensures that model play medical education assistant role in the whole process.
# if we keep ex["instruction"] as system , the model will play act what is asked in instructions that may change for each question.
# in our example, the instruction is same in all rows so we will keep the system_prompt as it is .
SYSTEM_PROMPT = (
    "You are a helpful medical education assistant. "
    "Answer accurately and concisely. You are not a substitute for a doctor."
)

def to_messages(ex):
    user_text = ex["instruction"]
    if ex["input"] and ex["input"].strip():
        user_text += "\n\n" + ex["input"]
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": ex["output"]},
    ]}

# filtering rows where output column is msising or output column has nothing in it.
raw = raw.filter(lambda x: x["output"] and len(x["output"].strip()) > 0)
raw = raw.shuffle(seed=42).select(range(NUM_SAMPLES))

# apply to_messages() to every row and deleting old columns as they have been now are in shape of to_messages()
dataset = raw.map(to_messages, remove_columns=raw.column_names)
print(dataset[0]["messages"])

Filter:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

[{'content': 'You are a helpful medical education assistant. Answer accurately and concisely. You are not a substitute for a doctor.', 'role': 'system'}, {'content': 'Answer this question truthfully\n\nWhat type of skin rash is frequently observed in patients with sarcoidosis?', 'role': 'user'}, {'content': 'Erythema nodosum is a type of skin rash that is commonly found in patients with sarcoidosis.', 'role': 'assistant'}]


In [7]:
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(len(train_ds), len(eval_ds))

4750 250


## Cell 6: Load tokenizer and model in full precision
No quantization and no adapters. We load in **float32** so the master weights are stable, and use fp16 *mixed precision* during training for speed.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#• Many causal language models (like Qwen or GPT) do not have a built-in padding token because they generate text continuously. However, during training, batch processing requires
# all sentences to be the exact same length.
# This line says: "If there isn't a padding token, just use the End-of-Sentence (eos_token) as a spacer to fill out short text."
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, device_map="auto"
)
model.config.use_cache = False

#Loops through every single layer in the model and counts the total number of parameters (weights).
n_params = sum(p.numel() for p in model.parameters())

#Counts only the parameters where requires_grad=True. In deep learning, if a parameter requires a gradient, it means it will be actively updated/changed by the trainer.
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params/1e6:.0f}M | Trainable: {n_train/1e6:.0f}M ({100*n_train/n_params:.0f}%)")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total params: 494M | Trainable: 494M (100%)


## Cell 7: Baseline test (BEFORE fine-tuning)

In [9]:
def ask(question, model, max_new_tokens=200):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question}]

    # converts the list of messages into the specific string format the model was trained to
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_questions = [
    "What is the mechanism of action of metformin?",
    "What are the classic symptoms of diabetic ketoacidosis?",
]
for q in test_questions:
    print("Q:", q); print("A:", ask(q, model)); print("=" * 60)

Q: What is the mechanism of action of metformin?
A: Metformin works by inhibiting the enzyme glucose-6-phosphatase, which breaks down glucose into simpler molecules like glycogen or glucose-6-phosphate. This leads to an increase in blood sugar levels, helping to lower them. Metformin also helps regulate insulin sensitivity, reducing the body's response to insulin and preventing excess fat storage.
Q: What are the classic symptoms of diabetic ketoacidosis?
A: Classic symptoms of diabetic ketoacidosis include:

1. Extreme thirst
2. Frequent urination (polyuria)
3. Fatigue or weakness
4. Nausea, vomiting
5. Diarrhea
6. Confusion or disorientation
7. Seizures
8. Coma

These symptoms can be severe and require immediate medical attention to prevent life-threatening complications.


## Cell 8: Training arguments
Key differences from LoRA: the learning rate is **~10x lower** (2e-5 instead of 2e-4), because every weight moves.

Out-of-memory? Lower `max_length` to 384 or batch size to 1 (keep the effective batch of 16 by raising accumulation to 16).

In [15]:
from huggingface_hub import login
login()   # paste your write token in the box that appears

In [19]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    optim="adamw_torch",
    fp16=True,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    max_length=512,
    report_to="none",

    # NEW: Hub upload settings
    push_to_hub=True,
    hub_model_id="NajafAli01/qwen-medical-full-sft",
    hub_private_repo=True,      # set to False if you want it public
)

## Cell 9: Create the trainer
No `peft_config` is passed, so all weights are trained.

In [20]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

In [21]:
train_result = trainer.train()
print(train_result.metrics)

 model is not performing as i wanted it to perform so we have to change its epoch, learning rate or etc to get better results in validation loss.

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.536608,1.008411,0.701173,0.776887,109000.000000
100,0.798418,0.974266,0.818523,0.781984,217854.000000
150,0.884510,0.932217,0.846072,0.785504,325100.000000
200,0.902952,0.914046,0.873438,0.788281,437087.000000
250,0.904341,0.898563,0.858687,0.793848,551049.000000
300,0.751846,0.882343,0.820200,0.796335,659743.000000
350,0.493050,0.927624,0.652392,0.793072,773770.000000
400,0.491042,0.925241,0.638424,0.793709,882054.000000
450,0.473780,0.920294,0.633510,0.795658,992374.000000
500,0.472809,0.918221,0.627457,0.797237,1100112.000000


{'train_runtime': 3007.1656, 'train_samples_per_second': 3.159, 'train_steps_per_second': 0.198, 'total_flos': 3506969156680704.0, 'train_loss': 0.6438807566157897, 'epoch': 2.0}


## Cell 11: Plot loss curves
If eval loss rises while train loss keeps falling, the model is overfitting; reduce epochs.

In [ ]:
import matplotlib.pyplot as plt
logs = trainer.state.log_history
plt.figure(figsize=(8, 4))
plt.plot([l["step"] for l in logs if "loss" in l], [l["loss"] for l in logs if "loss" in l], label="train")
plt.plot([l["step"] for l in logs if "eval_loss" in l], [l["eval_loss"] for l in logs if "eval_loss" in l], label="eval", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Cell 12: Save the full model
This is a standalone model (about 1-2 GB), not a small adapter.

In [ ]:
trainer.push_to_hub(commit_message="Full SFT on medical flashcards")

## Cell 13: Test AFTER fine-tuning

In [ ]:
for q in test_questions + ["What is the first-line treatment for hypertension in a diabetic patient?"]:
    print("Q:", q); print("A:", ask(q, trainer.model)); print("=" * 60)

## Cell 14 (optional): Reload the saved model later

In [ ]:
reloaded = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.float16, device_map="auto")
print(ask("Explain what an ACE inhibitor does.", reloaded))